In [1]:
import sys
from pathlib import Path
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

try:
    PROJECT_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from he.tenseal_context import create_context
from he.fhe_inference import encrypted_inference_demo

### 1. Load Test Data

In [2]:
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
test_images = torch.load(DATA_DIR / 'test_images.pt')
test_labels = torch.load(DATA_DIR / 'test_labels.pt')

print(f"Total test images: {len(test_images)}")

Total test images: 3589


### 2. Setup TenSEAL Context
Using `poly_modulus_degree=8192` for optimized 32x32 input size.

In [3]:
context = create_context()
print("TenSEAL Context created.")

TenSEAL Context created.


### 3. Run Accuracy Test
Running encrypted inference on a subset of the test data (e.g., 20 samples) because FHE inference is slow.

In [4]:
# Number of samples to test
NUM_SAMPLES = 20 
indices = range(NUM_SAMPLES)

correct_plain = 0
correct_enc = 0
total = 0
results = []

print(f"Starting inference on {NUM_SAMPLES} samples...")
start_time = time.time()

for idx in tqdm(indices):
    # Run inference (logs are reduced now)
    # We capture the result dict to check correctness programmatically
    res = encrypted_inference_demo(context=context, sample_index=idx, use_packed=True)
    
    true_label = res['true_label']
    plain_pred = res['plain_pred']
    enc_pred = res['encrypted_pred']
    
    if plain_pred == true_label:
        correct_plain += 1
    if enc_pred == true_label:
        correct_enc += 1
    
    total += 1
    results.append(res)

end_time = time.time()
duration = end_time - start_time

print(f"\n--- Test Complete ---")
print(f"Time taken: {duration:.2f}s ({duration/total:.2f}s per sample)")
print(f"Plain Model Accuracy: {correct_plain}/{total} ({correct_plain/total*100:.2f}%)")
print(f"Encrypted Model Accuracy: {correct_enc}/{total} ({correct_enc/total*100:.2f}%)")

Starting inference on 20 samples...


  0%|          | 0/20 [00:00<?, ?it/s][INFO] Loading model weights from /Users/hyunwookkim/Documents/study/25y2s/보안프로젝트설계/프로젝트/프로토타입/learning_test_01/fhe_emotion/models/fhe_cnn_fer2013_2.pt
[INFO] Using packed TenSEAL runner for inference
[INFO] Plain inference complete
[INFO] Plain pred: 0 | Encrypted pred: 0
  5%|▌         | 1/20 [00:17<05:28, 17.29s/it][INFO] Loading model weights from /Users/hyunwookkim/Documents/study/25y2s/보안프로젝트설계/프로젝트/프로토타입/learning_test_01/fhe_emotion/models/fhe_cnn_fer2013_2.pt
[INFO] Using packed TenSEAL runner for inference
[INFO] Plain inference complete
[INFO] Plain pred: 1 | Encrypted pred: 1
 10%|█         | 2/20 [00:34<05:11, 17.30s/it][INFO] Loading model weights from /Users/hyunwookkim/Documents/study/25y2s/보안프로젝트설계/프로젝트/프로토타입/learning_test_01/fhe_emotion/models/fhe_cnn_fer2013_2.pt
[INFO] Using packed TenSEAL runner for inference
[INFO] Plain inference complete
[INFO] Plain pred: 1 | 


--- Test Complete ---
Time taken: 327.60s (16.38s per sample)
Plain Model Accuracy: 9/20 (45.00%)
Encrypted Model Accuracy: 11/20 (55.00%)


### 4. Analysis
Check if there are any discrepancies between Plain and Encrypted predictions.

In [7]:
discrepancies = [r for r in results if r['plain_pred'] != r['encrypted_pred']]

if len(discrepancies) == 0:
    print("✅ Great! Plain and Encrypted predictions match perfectly for all tested samples.")
else:
    print(f"⚠️ Found {len(discrepancies)} discrepancies:")
    for r in discrepancies:
        print(f"True: {r['true_label']} | Plain: {r['plain_pred']} | Encrypted: {r['encrypted_pred']}")

⚠️ Found 4 discrepancies:
True: 3 | Plain: 6 | Encrypted: 3
True: 1 | Plain: 6 | Encrypted: 3
True: 3 | Plain: 6 | Encrypted: 0
True: 6 | Plain: 3 | Encrypted: 6
